# ⚡ Chạy Backtest Siêu Tốc Trên Google Colab Pro (High-RAM)
### Chiến Lược: `NostalgiaForInfinityX8_Combined_AI` + AI Gatekeeper v3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tminhnhat/NostalgiaForInfinity/blob/main/notebooks/Run_NFIX8_Backtest_Colab_HighRAM.ipynb)

---
### 🎯 Lợi Thế Khi Dùng Google Colab Pro Cho Backtest:
| Hạng mục | Máy tính cá nhân (i5 Gen 3) | Google Colab Pro (CPU High-RAM) |
| :--- | :--- | :--- |
| **Vi xử lý (CPU)** | Intel Core i5-3230M (2 nhân 4 luồng, Gen 3) | **8 vCPUs (Intel Xeon / AMD EPYC, AVX-512)** |
| **Bộ nhớ RAM** | 8 GB DDR3 (trống ~2.5 GB) | **~52 GB - 54 GB System RAM** |
| **Tốc độ Backtest (185 ngày)** | ~28 phút (chậm, nghẽn RAM) | **~3 - 5 phút (nhanh gấp 7 - 8 lần)** 🚀 |
| **Băng thông tải nến** | Mạng gia đình (~30 - 80 Mbps) | **Băng thông trung tâm dữ liệu (~1,000 Mbps)** |
| **Chi phí Compute Units (CU)** | Miễn phí điện máy tính | **Cực rẻ (~0.08 - 0.1 CU/giờ, chạy hàng trăm lần)** |

> 💡 **LƯU Ý CỰC KỲ QUAN TRỌNG VỀ CẤU HÌNH RUNTIME:**
> 1. Vào menu **Runtime (Thời gian chạy)** ➔ **Change runtime type (Thay đổi loại thời gian chạy)**.
> 2. **Hardware accelerator (Bộ tăng tốc)**: Chọn **None** (CPU).
>    *(Vì Freqtrade Backtest tính toán chỉ báo thuần trên CPU, không dùng GPU. Chọn None CPU giúp tiết kiệm 95% Compute Units, chạy cả tháng không lo hết điểm)*.
> 3. **Runtime shape (Cấu hình thời gian chạy)**: Chọn **High-RAM (RAM dung lượng cao)**.
>    *(Colab Pro sẽ cấp ngay 8 vCPUs và ~52GB RAM để xử lý mượt mà 100+ cặp nến Binance Futures)*.
> 4. Bấm **Save (Lưu)**.

In [ ]:
# ==============================================================================
# BƯỚC 1: KIỂM TRA PHẦN CỨNG HỆ THỐNG & TÀI NGUYÊN COLAB PRO
# ==============================================================================
import os
import psutil

vcpus = os.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
disk_gb = psutil.disk_usage('/').total / (1024 ** 3)
disk_free_gb = psutil.disk_usage('/').free / (1024 ** 3)

print(f"🖥️  Số luồng CPU (vCPUs): {vcpus} cores")
print(f"🧠  Dung lượng RAM:       {ram_gb:.1f} GB")
print(f"💾  Ổ đĩa lưu trữ:       {disk_free_gb:.1f} GB trống / {disk_gb:.1f} GB")

if ram_gb >= 30:
    print("\n✅ TUYỆT VỜI! Bạn đang chạy trên môi trường High-RAM (52GB RAM, 8 vCPUs)!")
else:
    print("\n⚠️  LƯU Ý: Bạn đang ở gói Standard RAM (~12GB). Hãy vào Runtime -> Change runtime type -> Chọn High-RAM để đạt tốc độ tối đa 3-5 phút!")

In [ ]:
# ==============================================================================
# BƯỚC 2: CÀI ĐẶT TA-LIB C-LIBRARY VÀ FREQTRADE (SIÊU TỐC ~1-2 PHÚT)
# ==============================================================================
import sys
import subprocess

print("⏳ Đang kiểm tra và cài đặt thư viện kỹ thuật TA-Lib C...")
install_talib_script = """
if [ ! -f /usr/lib/libta_lib.so.0 ]; then
    wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
    tar -xzf ta-lib-0.4.0-src.tar.gz
    cd ta-lib/
    ./configure --prefix=/usr > /dev/null 2>&1
    make -j$(nproc) > /dev/null 2>&1
    sudo make install > /dev/null 2>&1
    cd ..
    rm -rf ta-lib ta-lib-0.4.0-src.tar.gz
    echo '✅ Đã cài đặt xong TA-Lib C library!'
else
    echo '✅ TA-Lib C library đã sẵn sàng!'
fi
"""
subprocess.run(install_talib_script, shell=True, check=True, executable='/bin/bash')

print("⏳ Đang cài đặt Freqtrade, LightGBM và các gói phụ thuộc...")
!pip install -q ta-lib freqtrade lightgbm python-rapidjson pyarrow

# Xác minh cài đặt thành công
import talib
import freqtrade
import lightgbm
import rapidjson
print(f"\n🎉 XÁC NHẬN SẴN SÀNG:")
print(f"   - Freqtrade: {freqtrade.__version__}")
print(f"   - TA-Lib:    {talib.__version__}")
print(f"   - LightGBM:  {lightgbm.__version__}")

In [ ]:
# ==============================================================================
# BƯỚC 3: CLONE HOẶC CẬP NHẬT MÃ NGUỒN BOT MỚI NHẤT TỪ GITHUB
# ==============================================================================
import os
from pathlib import Path

REPO_URL = "https://github.com/tminhnhat/NostalgiaForInfinity.git"
REPO_DIR = "/content/NostalgiaForInfinity"

if not os.path.exists(REPO_DIR):
    print(f"📥 Đang clone repository từ: {REPO_URL}...")
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"🔄 Đang cập nhật repository mới nhất...")
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}

# Kiểm tra xác nhận file chiến lược & mô hình AI
assert Path("NostalgiaForInfinityX8_Combined_AI.py").exists(), "Thiếu file chiến lược!"
assert Path("configs/backtest_config.json").exists(), "Thiếu file config backtest!"
assert Path("user_data/models/nfix8_short_ai_v3.pkl").exists(), "Thiếu file mô hình AI Gatekeeper v3!"

print("\n✅ Thư mục làm việc hiện tại:", os.getcwd())
print("✅ Chiến lược & Mô hình AI Gatekeeper v3 đã sẵn sàng!")

In [ ]:
# ==============================================================================
# BƯỚC 4: TẢI HOẶC ĐỒNG BỘ DỮ LIỆU NẾN 185 NGÀY (BINANCE FUTURES)
# ==============================================================================
import os
from pathlib import Path

# Tùy chọn 1: Nếu anh muốn lưu cache nến vào Google Drive để không phải tải lại mỗi lần:
MOUNT_GOOGLE_DRIVE = False  # Đổi thành True nếu muốn gắn Google Drive

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA_DIR = Path('/content/drive/MyDrive/freqtrade_data/binance/futures')
    LOCAL_DATA_DIR = Path('/content/NostalgiaForInfinity/user_data/data/binance/futures')
    LOCAL_DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_DATA_DIR.exists():
        print("⚡ Đang copy nhanh dữ liệu từ Google Drive...")
        !cp -r {DRIVE_DATA_DIR} {LOCAL_DATA_DIR.parent}/

# Tùy chọn 2: Tải trực tiếp từ Binance Futures (Băng thông Colab 1Gbps chỉ mất ~1-2 phút)
print("📥 Đang tải dữ liệu 185 ngày từ sàn Binance Futures...")
!freqtrade download-data \
    --config configs/backtest_config.json \
    --days 185 \
    --timeframes 5m 15m 1h 4h 1d \
    --trading-mode futures \
    --data-format-ohlcv feather

print("\n🎉 Tải dữ liệu hoàn tất! Kiểm tra thư mục dữ liệu:")
!ls -lh user_data/data/binance/futures | head -n 10

In [ ]:
# ==============================================================================
# BƯỚC 5: CHẠY BACKTEST ĐÁNH GIÁ SIÊU TỐC (8 THREADS, ~3 - 5 PHÚT)
# ==============================================================================
import time

start_time = time.time()
print("🚀 BẮT ĐẦU CHẠY BACKTEST TRÊN 8 vCPUs (HIGH-RAM)...")

!python scripts/run_eval_backtest.py \
    --strategy NostalgiaForInfinityX8_Combined_AI \
    --days 185 \
    --run-label "Colab_Pro_HighRAM"

elapsed = time.time() - start_time
print(f"\n⏱️ TỔNG THỜI GIAN THỰC THI: {elapsed / 60:.2f} phút ({elapsed:.1f} giây)!")

In [ ]:
# ==============================================================================
# BƯỚC 6: HIỂN THỊ BÁO CÁO HIỆU SUẤT TRỰC QUAN (MARKDOWN FORMATTED)
# ==============================================================================
from IPython.display import display, Markdown
from pathlib import Path

report_file = Path("reports/latest_report.md")
if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        content = f.read()
    display(Markdown(content))
else:
    print("⚠️ Không tìm thấy file reports/latest_report.md!")

In [ ]:
# ==============================================================================
# BƯỚC 7: TẢI BÁO CÁO & KẾT QUẢ VỀ MÁY TÍNH CỦA BẠN
# ==============================================================================
from google.colab import files
import glob

print("📥 Đang chuẩn bị tệp tải xuống...")
if Path("reports/latest_report.md").exists():
    files.download("reports/latest_report.md")

# Tải file JSON kết quả chi tiết mới nhất
history_files = sorted(glob.glob("reports/history/backtest_*.json"))
if history_files:
    latest_history = history_files[-1]
    print(f"📥 Đang tải kết quả JSON: {latest_history}")
    files.download(latest_history)

print("✅ Hoàn thành!")